# Fashion-MNIST: TensorFlow CNN vs Dense MLP

**Question:** Do convolutional layers improve clothing recognition under a fixed five-epoch budget?

This executed notebook inspects the completed experiment and recalculates its scores. Training was performed by `train.py`; this notebook does not refit the models. Automated authorship; human review remains pending.

Cells were executed sequentially with IPython in-process; all displayed values come from the recorded experiment.

In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, accuracy_score, balanced_accuracy_score

root = Path.cwd()
metrics = json.loads((root / 'metrics.json').read_text())
audit = json.loads((root / 'audit.json').read_text())
config = json.loads((root / 'study_config.json').read_text())
labels = pd.read_csv(root / 'class_names.csv').set_index('class_id')['label']
print('Selected model:', metrics['selected_model'])
print('Seed:', config['seed'], '| Epochs:', config['epochs'])

Selected model: CNN
Seed: 42 | Epochs: 5


## Source and quality

The four pinned source archives match the author-published checksums. Every decompressed payload was validated. The 10,000 official test images stay separate from the stratified training folds. No exact duplicate images or cross-partition overlaps were found.

![Data quality](figures/data-quality.png)

In [2]:
pd.DataFrame({'class': labels, 'training_images': audit['training_class_counts'], 'test_images': audit['test_class_counts']})

Out[2]: 
                class  training_images  test_images
class_id                                           
0         T-shirt/top             6000         1000
1             Trouser             6000         1000
2            Pullover             6000         1000
3               Dress             6000         1000
4                Coat             6000         1000
5              Sandal             6000         1000
6               Shirt             6000         1000
7             Sneaker             6000         1000
8                 Bag             6000         1000
9          Ankle boot             6000         1000


## Training comparison

Each architecture was fitted separately on each of three training folds for five epochs. Both models use in-model rescaling, Adam, and dropout. The chosen metric is macro-F1, giving equal weight to each clothing category. The best mean CV score selected the CNN before final test scoring.

In [3]:
pd.DataFrame(metrics['leaderboard'])[['model', 'cv_mean', 'cv_std', 'fold_scores']]

Out[3]: 
            model  ...                                        fold_scores
0             CNN  ...  [0.8829929977503692, 0.8832968197091905, 0.880...
1       Dense MLP  ...  [0.8696755309687946, 0.8645217364348007, 0.866...
2  Dummy baseline  ...  [0.01818181818181818, 0.01818181818181818, 0.0...

[3 rows x 4 columns]


## CNN architecture

The CNN preserves local pixel neighborhoods through two convolution and pooling stages. The dense comparator flattens all 784 pixels. These architecture choices were fixed before measuring results.

In [4]:
print((root / 'model_summary.txt').read_text())

Model: "CNN"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 28, 28, 16)     │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 14, 14, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 14, 14, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 7, 7, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───

## Official holdout evaluation

The final CNN was fitted on all 60,000 training images. The following cell checks the reported metrics against all saved test predictions.

In [5]:
predictions = pd.read_csv(root / 'predictions.csv')
y = predictions['actual']
yhat = predictions['predicted']
recalculated = {'macro_f1': f1_score(y, yhat, average='macro'), 'accuracy': accuracy_score(y, yhat), 'balanced_accuracy': balanced_accuracy_score(y, yhat)}
assert recalculated == metrics['test_metrics'][metrics['selected_model']]
assert len(predictions) == 10000
assert (~predictions['correct']).sum() == metrics['test_errors']
pd.DataFrame(metrics['test_metrics']).T

Out[5]: 
                macro_f1  accuracy  balanced_accuracy
CNN             0.887105    0.8874             0.8874
Dummy baseline  0.018182    0.1000             0.1000


## Where the model fails

Inspect complete error counts by class pair instead of choosing attractive examples. The figure shows the first twelve errors by source row. Confidence is an uncalibrated softmax score.

![Confusion matrix](figures/confusion-matrix.png)

![Error examples](figures/error-examples.png)

In [6]:
errors = predictions.loc[~predictions['correct']].copy()
errors['actual_label'] = errors['actual'].map(labels)
errors['predicted_label'] = errors['predicted'].map(labels)
errors.groupby(['actual_label', 'predicted_label']).size().sort_values(ascending=False).head(10).rename('errors').to_frame()

Out[6]: 
                              errors
actual_label predicted_label        
Shirt        T-shirt/top         136
Pullover     Coat                110
Shirt        Coat                 95
T-shirt/top  Shirt                89
Pullover     Shirt                87
Coat         Shirt                58
Shirt        Pullover             55
Coat         Pullover             45
Dress        Coat                 40
Sneaker      Sandal               38


In [7]:
pd.DataFrame({name: scores for name, scores in metrics['per_class'].items() if name in labels.values}).T.sort_values('f1-score')

Out[7]: 
             precision  recall  f1-score  support
Shirt         0.713531   0.675  0.693731   1000.0
Coat          0.764184   0.862  0.810150   1000.0
Pullover      0.863181   0.776  0.817272   1000.0
T-shirt/top   0.828909   0.843  0.835895   1000.0
Dress         0.877432   0.902  0.889546   1000.0
Sneaker       0.954733   0.928  0.941176   1000.0
Ankle boot    0.960922   0.959  0.959960   1000.0
Sandal        0.951644   0.984  0.967552   1000.0
Bag           0.979859   0.973  0.976417   1000.0
Trouser       0.986802   0.972  0.979345   1000.0


## Learning curves and limits

![Mean training and validation curves](figures/learning-curves.png)

Validation uses the final weights for each epoch with dropout disabled; training metrics accumulate during updates with dropout active. That helps explain why validation accuracy can exceed training accuracy. Five epochs are a bounded comparison, not proof of convergence or optimal performance. Near duplicates and shared product identities cannot be ruled out, and these low-resolution benchmark images do not establish real-world photo performance.

In [8]:
verification = json.loads((root / 'verification.json').read_text())
print('Verification:', verification['status'])
print('All 10,000 reloaded-weight predictions match:', verification['reloaded_weight_predictions_match_all_test_rows'])
print('Maximum probability difference:', verification['max_probability_difference_after_weight_reload'])

Verification: passed
All 10,000 reloaded-weight predictions match: True
Maximum probability difference: 0.0


## Human review

[Learning notes](LEARNING_NOTES.md) are intentionally unanswered for the human author. Dataset credit and licensing are recorded in [SOURCE.md](SOURCE.md).